In [1]:
from pathlib import Path

import pandas as pd
from impc_api import solr_request, batch_solr_request

In [2]:
# Create output directory.
output_dir = Path("lap_eap_activity_data")
output_dir.mkdir(parents=True, exist_ok=True)

# Request list of available phenotyping centers in the experiment core.
num_found, df_centers = solr_request(
    core='experiment',
    params={
        'q': 'procedure_stable_id:*OFD* AND life_stage_name:"Late adult"',
        'facet': 'on',
        'facet.field': 'phenotyping_center',
        'facet.limit': 20,
        'facet.mincount': 1,
    },
    silent=True
)

# Convert facet query result to list of available centers.
centers = list(df_centers["phenotyping_center"])

# Remove "RBRC" from the list of centers.
centers.remove("RBRC")
print(centers)

# Create query.
centers_query = " OR ".join(f'phenotyping_center:"{c}"' for c in centers)

['MRC Harwell', 'BCM', 'JAX', 'KMPC', 'HMGU', 'ICS']


In [3]:
# Function to convert list of "key = value" into dict.
def parse_metadata(meta_list):
    return dict(item.split(" = ", 1) for item in meta_list)

# Function to expand metadata.
def expand_metadata(df):
    meta_df = df["metadata"].apply(parse_metadata).apply(pd.Series)
    df = df.drop(columns=["metadata"]).join(meta_df)
    return df

procedures = ["OFD", "GRS", "DXA", "CSD"]
for procedure in procedures:
    # To get locomotor activity, query by parameter was used. 
    # It was manually verified that all *_032_001 entries correspond 
    # to locomotor activity parameters for the specified list of centers.
    if procedure == "CSD":
        query= f'({centers_query}) AND parameter_stable_id:*{procedure}_032_001'
    else:
        query = f'({centers_query}) AND procedure_stable_id:*{procedure}*'

    # Experimental data, late adult and middle aged adult.
    df = batch_solr_request(
        core='experiment',
        params={
            'q':f'{query} AND (life_stage_name:"Late adult" OR life_stage_name:"Middle aged adult") AND biological_sample_group:experimental',
        },
        download=False,
        batch_size=10000
    )

    # Generate list of Late adult alleles.
    allele_list = df["allele_accession_id"].unique()
    print(f"Number of LA alleles for {procedure}: {len(allele_list)}")

    # Experimental data, early adult.
    alleles_without_ea_data = list()
    for allele_id in allele_list:
        ea_query = f'{query} AND allele_accession_id:"{allele_id}" AND (life_stage_name:"Early adult")'

        num_found, _ = solr_request(
            core='experiment',
            params={
                'q':f'{ea_query}',
            },
            silent=True
        )

        if num_found == 0:
            # Add allele_accession_id to the list of data with no EA.
            alleles_without_ea_data.append(allele_id)
            
        else:
           # Request EA data. 
            df_allele = batch_solr_request(
                core='experiment',
                params={
                    'q':f'{ea_query}',
                },
                download=False
            )
            df = pd.concat([df, df_allele], axis=0, ignore_index=True)

    # Remove LA data from the dataset.
    df = df[~df["allele_accession_id"].isin(alleles_without_ea_data)]

    # Expand metadata column.
    df = expand_metadata(df)

    # Save experimental data.
    df.to_csv(output_dir / Path(f"{procedure}_exp.zip"), index=False, compression={"method": "zip", "archive_name": f"{procedure}_exp.csv"})

    # Control data.
    df_controls = batch_solr_request(
        core='experiment',
        params={
            'q':f'{query} AND (life_stage_name:"Late adult" OR life_stage_name:"Middle aged adult" OR life_stage_name:"Early adult") AND biological_sample_group:control',
        },
        download=False
    )

    # Expand metadata column.
    df_controls = expand_metadata(df_controls)
    
    # Save experimental data.
    df_controls.to_csv(output_dir / Path(f"{procedure}_ctrl.zip"), index=False, compression={"method": "zip", "archive_name": f"{procedure}_ctrl.csv"})

Number of found documents: 291486


300000it [05:03, 989.37it/s]                                                                                                                                          


Number of LA alleles for OFD: 376
Number of found documents: 198


5000it [00:00, 8844.92it/s]                                                                                                                                           


Number of found documents: 322


...

5000it [00:00, 75132.54it/s]                                                                                                                                          


Number of found documents: 16


5000it [00:00, 16832.47it/s]                                                                                                                                          


Number of found documents: 28669


30000it [00:32, 925.58it/s]                                                                                                                                           
